# Stage 2 Notebook 27 - Exp2V Grid detection head + query lane head

**Why this exists.** Companion to Exp2U (lane-only diagnostic). Same hypothesis: detection has been sabotaging lane training because DETR with 100 queries cannot converge in 10 epochs, producing large but non-converging gradient flow into the shared backbone (`mAP50 = 0.003-0.005` across every Exp2 run). Exp2U tests whether removing detection helps. Exp2V tests whether *making detection actually work* helps.

Single-knob change vs Exp2P: `model.detection_head.type: detr -> grid`. The codebase already has `SimpleVehicleDetectionHead` (anchor-free single-shot grid head) and `SimpleVehicleDetectionLoss`. A grid head can converge in our 10-epoch budget where DETR cannot.

Decision tree at epoch 10:
- `mAP50 >= 0.10` AND `decoded_f1 >= 0.10`: simpler det head converges AND helps lane. The path forward is grid det + query lane jointly.
- `mAP50 >= 0.10` BUT `decoded_f1 ~ 0.03`: det converges but doesn't help lane. Detection convergence isn't the issue; multi-task interference is structural.
- `mAP50 < 0.05`: even the simpler head doesn't converge. Detection task itself is too hard at this resolution / dataset size.

Independent of Exp2U. Run either or both.

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After smoke + debug pass, change to `False` for the 10-epoch short run.
3. Output mirrored to notebook cell, Colab runtime log, Drive log file.
4. Do not rerun NB00.

In [3]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [4]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp22_rmt_gca_grid_det_with_query_lane_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp22_rmt_gca_grid_det_with_query_lane_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp22_rmt_gca_grid_det_with_query_lane_joint_smoke.log
Traceback (most recent call last):
  File "/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/scripts/smoke_test_joint_models.py", line 76, in <module>
    main()
  File "/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/scripts/smoke_test_joint_models.py", line 72, in main
    run_one(Path(item))
  File "/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/scripts/smoke_test_joint_models.py", line 54, in run_one
    det_loss, det_comp = det_loss_fn(out['det'], [torch.tensor([[0, 0.5, 0.5, 0.2, 0.2]], dtype=torch.float32)])
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torc

CalledProcessError: Command '['/usr/bin/python3', '-u', 'stage2/scripts/smoke_test_joint_models.py', 'stage2/configs/exp22_rmt_gca_grid_det_with_query_lane_joint.yaml']' returned non-zero exit status 1.

In [ ]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp22_rmt_gca_grid_det_with_query_lane_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short10'
    EPOCHS = 10
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

## What to watch in Exp2V training

Reference Exp2P (queries, DETR det): `mAP50=0.004`, `decoded_f1=0.026`, `matched_iou=0.13`.

Pass criteria at epoch 10:
- **`val/det/metric_map50 >= 0.10`**: grid head converges where DETR couldn't. Critical sanity check.
- **`val/lane/decoded_f1 >= 0.10`**: lane benefits from a converging detection task (proper backbone signal).
- **`val/matched_line_iou >= 0.25`**: geometry recovers because backbone isn't being pulled by random detection gradient.
- `val_lane_f1 >= 0.55`.

Failure modes:
- mAP50 stays < 0.05: grid head also can't converge at this resolution / dataset. The detection task itself is too hard.
- mAP50 converges but lane stays stuck: detection convergence isn't the saboteur; the multi-task interference is structural (gradient direction conflict, not magnitude). Try gradient surgery (PCGrad) or task-specific backbones.

Compare with Exp2U (NB26):
- If Exp2U has high decoded_f1 but Exp2V doesn't: detection is poison regardless of head choice.
- If Exp2V has high decoded_f1 (matching Exp2U): a working detection head IS compatible with a working lane head.
- If neither: detection isn't the issue; the lane impasse is something else.